In [91]:
#### IMPORTS
import json
from time import time
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt

In [92]:
Uc = 1.0
Lc = 2*np.pi
Re_list = np.linspace(10, 100, 19)
t_list = np.arange(0, 10.1, 0.5)

In [93]:
output = Path("output/")
N_snapshots = len(t_list)

# Load mesh (centroids, vertices, faces)

In [94]:
# Mesh
with h5py.File("mesh.h5", "r") as file:
    mesh_centers = file["centroids"][:].astype(np.float32) # I computed these from 0 to 1, let's rescale
    mesh_vertices = file["vertices"][:].astype(np.float32) # I computed these from 0 to 1, let's rescale
    mesh_indices = file["indices"][:].astype(np.uint32)

mesh_centers *= Lc
mesh_vertices *= Lc
N_cells = len(mesh_centers)

# Save dataset data

In [95]:
output.mkdir(exist_ok=True)
# copy mesh file to output

with h5py.File(output / "mesh.h5", "w") as file:
    file.create_dataset("centroids", data=mesh_centers, compression="gzip", chunks=True)
    file.create_dataset("vertices", data=mesh_vertices, compression="gzip", chunks=True)
    file.create_dataset("indices", data=mesh_indices, compression="gzip", chunks=True)

data = {
    "solver": "analytical",
    "Uc": Uc,
    "Lc": Lc,
    "minRe": 10,
    "maxRe": 100
}
with open(output / "constants.json", "w") as file:
    json.dump(data, file, indent=4)

# Generate Taylor-Green vortex data

In [96]:
for j, Re in enumerate(Re_list):
    nu = Uc*Lc/Re

    U = np.zeros((N_snapshots, N_cells, 2), dtype=np.float32)
    p = np.zeros((N_snapshots, N_cells), dtype=np.float32)

    for i, t in enumerate(t_list):
        U[i,:,0] = np.sin(mesh_centers[:,0]) * np.cos(mesh_centers[:,1])*np.exp(-2*nu*t)
        U[i,:,1] = -np.cos(mesh_centers[:,0]) * np.sin(mesh_centers[:,1])*np.exp(-2*nu*t)
        p[i,:] = 0.25 * (np.cos(2*mesh_centers[:,0]) + np.cos(2*mesh_centers[:,1])) * np.exp(-4*nu*t)

    with h5py.File(output / f"{j}.h5", "w") as file:
        file.attrs["nu"] = nu
        file.create_dataset("U", data=U, compression="gzip", chunks=(1,N_cells,2))
        file.create_dataset("p", data=p, compression="gzip", chunks=(1,N_cells))
        file.create_dataset("t", data=t_list, compression="gzip", chunks=(N_snapshots,))